# oscNext L4 — end to end (LightGBM)

Everything from L3 `.i3` files to the trained L4 classifiers.

**The method is the one in the technical note:** LightGBM with the Table 10
hyperparameters.  (The project ran pybdt/AdaBoost for a while; measured on the
same data it kept 65.8% of the signal at 99% noise rejection where LightGBM
kept 95.9%.  See `CLAUDE.md`.)

## Runtime environment

The kernel must be the python inside the IceTray build's `env-shell.sh`,
otherwise `icecube.*` cannot be imported.  See `README.md`.

## The heavy lifting lives in the package

This notebook is a thin interface.  Processing is `scripts/process_L4.py`,
reading is `oscnext_l4.data`, the driver is `oscnext_l4.runner`, training is
`scripts/train_L4_classifier.py`.  None of that logic is repeated here, so
there is exactly one implementation of each thing.


## 0. Configuration and environment check

`lightgbm` and `tables` are required.  `pandas` is **not used** — it may not
exist in the IceTray environment.


In [ ]:
# Reload edited modules without restarting the kernel.  Without this, a
# `git pull` leaves the old module object in memory and you get errors like
# "cannot import name ... from oscnext_l4.data" even though the file is fine.
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
    print("autoreload  ON   (edits to oscnext_l4/* are picked up)")
except Exception:
    pass

import os, sys, glob, json, shlex, subprocess, time
import numpy as np

# --- required: lightgbm ---
try:
    import lightgbm as lgb
    print("lightgbm     OK  ", lgb.__version__)
except ImportError as e:
    raise SystemExit(
        "lightgbm could not be imported (%s).\n"
        "Training is impossible without it.  From the IceTray environment:\n"
        "  ./setup_env.sh run python -c 'import lightgbm'" % e)

# --- required: pytables ---
try:
    import tables
    print("tables       OK  ", tables.__version__)
except ImportError:
    raise SystemExit("pytables is missing -- HDF5 cannot be read.")

# --- optional ---
_NOTE = {"matplotlib": "no plots",
         "simweights": "CORSIKA weights fall back to the manual formula",
         "ipywidgets": "progress bar degrades to ASCII"}
for _n in _NOTE:
    try:
        _m = __import__(_n)
        print("%-12s OK   %s" % (_n, getattr(_m, "__version__", "")))
    except ImportError:
        print("%-12s MISSING  (%s)" % (_n, _NOTE[_n]))

import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

In [ ]:
# ---------------------------------------------------------------------------
# PATHS
# ---------------------------------------------------------------------------
# This notebook lives in <repo>/notebooks/, but Jupyter's working directory
# may be either the repo root or the notebook directory -- and a Jupyter
# server started in the wrong place is the single most common failure here
# (a kernel restart does NOT change it).  So locate the root explicitly.
REPO_ROOT = os.environ.get("OSCNEXT_L4_ROOT")
if not REPO_ROOT:
    for _cand in (os.getcwd(), os.path.dirname(os.getcwd())):
        if os.path.isdir(os.path.join(_cand, "oscnext_l4")):
            REPO_ROOT = _cand
            break
if not REPO_ROOT:
    raise SystemExit(
        "Could not locate the repository root from cwd=%s.\n"
        "Start Jupyter from the repo (or its notebooks/ directory), or set\n"
        "  export OSCNEXT_L4_ROOT=/path/to/osncnextl4" % os.getcwd())
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Outputs.  The HDF5 files run to TENS OF GB, and where they go matters more
# than it looks:
#
#   /home on cobalt is a SHARED 9.5 TB filesystem sitting at ~94% full.  There
#   is no per-user quota (`quota -s` prints nothing), so the free space shown
#   there is everyone's -- it moves with other people's usage, you cannot
#   delete your way out of it, and filling it hurts the whole cluster.
#   /data/user/$USER is on ceph, with PETABYTES free.
#
# So default to /data/user/$USER when it exists and is writable.  An explicit
# OSCNEXT_OUT_ROOT still wins; this only decides what happens when nothing was
# said, which used to be "write tens of GB into the repository".
def _default_output_root():
    env = os.environ.get("OSCNEXT_OUT_ROOT")
    if env:
        return env
    data_user = os.path.join("/data/user", os.environ.get("USER", ""))
    if os.path.isdir(data_user) and os.access(data_user, os.W_OK):
        return os.path.join(data_user, "L4_output")
    print("[!] /data/user/$USER is not available -- writing under the repo.")
    print("    That is fine for a smoke test and not for a production run;")
    print("    set OSCNEXT_OUT_ROOT to somewhere with room.")
    return os.path.join(REPO_ROOT, "L4_output")


OUTPUT_ROOT = _default_output_root()

# WHICH PRODUCTION.  "pass3" or "pass2" -- this one word drives the GCD, the
# sample paths, the cleaned pulse series and the noise weight unit (section 1).
PRODUCTION = "pass2"

# Separate output trees, so the two productions never overwrite each other.
# pass3 keeps the bare names it has always used, so existing files stay put.
_SUF = "" if PRODUCTION == "pass3" else "_" + PRODUCTION

HDF_BASE  = os.path.join(OUTPUT_ROOT, "hdf5" + _SUF)     # process_L4.py output
DS_BASE   = os.path.join(OUTPUT_ROOT, "ds" + _SUF)       # .npz training sets
MODEL_DIR = os.path.join(OUTPUT_ROOT, "models" + _SUF)   # .txt + .json + plots

for _d in (HDF_BASE, DS_BASE, MODEL_DIR):
    os.makedirs(_d, exist_ok=True)

PROCESS_PY = os.path.join(REPO_ROOT, "scripts", "process_L4.py")
TRAIN_PY   = os.path.join(REPO_ROOT, "scripts", "train_L4_classifier.py")

RNG_SEED = 12345
rng = np.random.default_rng(RNG_SEED)

_st = os.statvfs(OUTPUT_ROOT)
print("repo root  : %s" % REPO_ROOT)
print("output root: %s" % OUTPUT_ROOT)
_free_gb = _st.f_bavail * _st.f_frsize / 1e9
print("free disk  : %.1f GB%s"
      % (_free_gb, "   [!] TIGHT -- a full run writes tens of GB"
         if _free_gb < 2000 else ""))
for _p in (PROCESS_PY, TRAIN_PY):
    print("%-24s %s" % (os.path.basename(_p),
                        "found" if os.path.exists(_p) else "MISSING!"))

## 1. L3 to L4 processing

Runs `scripts/process_L4.py` for each sample.  **This takes hours** and is
done once — after the HDF5 files exist you can resume from section 2.

`--apply-cut` is **not** used: before the models are trained every event must
be booked, otherwise we would be cutting the training set itself.


In [ ]:
# ---------------------------------------------------------------------------
# WHICH PRODUCTION, AND WHAT TO LOAD
# ---------------------------------------------------------------------------
# The facts live in config/productions.json -- it is PURE DATA, so reading it
# is json.load() and nothing else.  Every value is explained in
# config/README.md, which is where the reasons went when the tables became
# JSON; several of them fail SILENTLY when wrong, so read it before editing.
# OSCNEXT_L4_CONFIG points at a different file.
from oscnext_l4.data import set_noise_weight_unit

CONFIG_PATH = os.environ.get("OSCNEXT_L4_CONFIG") or os.path.join(
    REPO_ROOT, "config", "productions.json")
with open(CONFIG_PATH) as _fh:
    CONFIG = json.load(_fh)
print("production config: %s" % CONFIG_PATH)

# WHAT TO LOAD, said as ROLES rather than as sample names.  The roles are the
# same in both productions while the names are not, so this needs no
# per-production branch and picks up a set one production has and the other
# does not -- pass2's NuTau is signal and is included; pass3 has no NuTau set
# in these paths, so a pass3 run simply has one fewer signal sample.
#
# The L4 is TWO classifiers trained in order, and the order is not ours: the
# production trains the noise BDT first, applies it, and trains the muon BDT
# only on what survives (note sec. 3.6.3 -- its background is detector data
# "following L4 noise cut", which is what makes it 99% muon).  So this is a
# stage switch, not a preference.
STAGE = "noise"               # "noise" -> "muon" -> "all"

NOISE_BDT_ROLES = tuple(CONFIG["bdt_roles"]["noise"])
MUON_BDT_ROLES = tuple(CONFIG["bdt_roles"]["muon"])
ROLES = {"noise": NOISE_BDT_ROLES,   # signal + the noise background
         "muon":  MUON_BDT_ROLES,    # signal + the muon background
         "all":   None}[STAGE]       # everything, both classifiers

_prod = CONFIG["productions"][PRODUCTION]
GCD = _prod["gcd"]
CLEANED_PULSES = _prod.get("cleaned_pulses")

# THE NOISE WEIGHT UNIT IS SET HERE, BESIDE THE PRODUCTION AND NOT LATER.
# It is 1/ns at pass3 and already Hz at pass2, nothing raises when it is
# wrong, and being wrong scales every noise rate by 1e9 -- the only symptom is
# an absurd number much later, in the training.  Keeping it in the same cell
# as the switch is what makes "select pass2 and forget" impossible.
set_noise_weight_unit(_prod["noise_weight_unit"])

SAMPLES = {}
for _name, _spec in _prod["samples"].items():
    if ROLES and _spec["kind"] not in ROLES:
        continue
    _cfg = {k: v for k, v in _spec.items() if not k.startswith("__")}
    _cfg["flags"] = list(_cfg["flags"])
    if CLEANED_PULSES:
        _cfg["flags"] += ["--cleaned-pulses", CLEANED_PULSES]
    # Each sample gets its own subdirectory, so two productions never
    # overwrite each other.
    _cfg["hdf5"] = os.path.join(HDF_BASE, _name, "L4_%s.hdf5" % _name)
    SAMPLES[_name] = _cfg

print("production     : %s" % PRODUCTION)
print("  stage        : %s  (roles: %s)"
      % (STAGE, ", ".join(ROLES) if ROLES else "all"))
print("  GCD          : %s" % GCD)
if CLEANED_PULSES:
    print("  cleaned pulses: %s" % CLEANED_PULSES)
print("  HDF5 base    : %s\n" % HDF_BASE)

for _name, _cfg in SAMPLES.items():
    _pats = _cfg["l3"] if isinstance(_cfg["l3"], list) else [_cfg["l3"]]
    _n = sum(len(glob.glob(p)) for p in _pats)
    _cfg["n_l3_files"] = _n
    print("  %-8s %-10s %6d L3 files" % (_name, _cfg["kind"], _n))

_missing = [n for n, c in SAMPLES.items() if c["n_l3_files"] == 0]
if _missing:
    print("\n  [!] no L3 file found for: %s" % ", ".join(_missing))
    print("      check config/productions.json before running anything.")

In [ ]:
from oscnext_l4.runner import (configure_runner, run_process, run_all,
                               run_process_per_run)  # noqa: F401
configure_runner(SAMPLES, PROCESS_PY, GCD)

### Smoke test first

Before the full production, process 200 frames of a single file and confirm
the chain works.

> **`--n` counts FRAMES, not events.**  The stream carries G/C/D, Q and P
> frames; only some P frames match `InIceSplit` and pass the L3 cut.
> 200 frames landing on ~60 events is normal — the output shows each stage.


In [ ]:
smoke = run_process("nue", n_frames=200)

### Full production

`chunk_files=10` puts every 10 L3 files in their own part
(`L4_nue_part000.hdf5`, ...).

- **Real percentage and ETA** — the number of parts is known up front.
- **Resumable** — finished parts are skipped, and a crash loses only the part
  that was in flight.

Corrupt `.i3.zst` files are removed by `--scan quick` (the default); if a tray
still dies, `--retries` drops the offending file and continues.


In [ ]:
# jobs>1 -> each sample is processed in N parallel processes (the biggest
#           speedup).  cobalt is a SHARED machine: 8 is reasonable, 64 is not.
# run_optional defaults to False: I3TensorOfInertia and separation_in_cogs
#           are in neither Table 11 nor Table 12, so they are not BDT inputs
#           and are skipped.  Pass run_optional=True if you want them (they
#           only cost time).
# EVERYTHING, on purpose: inadequate noise statistics is this analysis's known
# bottleneck (open risk 5e -- 100 pass3 files gave ~2000 events, which is why
# nothing right of 99% rejection was measurable), and pass2 noise 888003 has
# 10000 L3 files.  Taking all of them is the point.
#
# max_files caps the L3 files per sample if you ever want a quick partial run:
#     run_all(jobs=8, chunk_files=10, max_files={"noise": 1000})
# It keeps the first N of the sorted list, so a capped run can be extended
# later without reprocessing what is already there.
#
# DETECTOR DATA NEEDS NOTHING SPECIAL HERE.  It has one GCD per run -- the
# averaged MC GCD cannot stand in, because the dead DOMs and the calibration
# are exactly what changes between runs, and a pass2 L3 data file carries no
# G/C/D frames of its own -- so it is routed to run_process_per_run
# automatically, on the SPEC (does this sample declare a `gcd` list?) rather
# than on the name.  One run per invocation, `jobs` of them at a time, output
# named after the run.
#
# For data, use `fraction` and NOT max_files.  Its `l3` is eighteen patterns
# ordered by year, so max_files would take all of 2012-2013 and none of 2017 --
# the seasonal bias the note's run list was chosen to avoid:
#     run_all(jobs=16, chunk_files=10, fraction={"data": 0.5})
results = run_all(jobs=8, chunk_files=10)

## 2. Booking check

**This is the first thing to do once processing finishes.**  If a tray module
failed silently its table is never written; catch that here rather than three
sections later while wondering why a variable is always NaN.


In [ ]:
from oscnext_l4.data import dump_tables, find_hdf5

# Not hardcoding "_smoke.hdf5": if the smoke test was never run that file does
# not exist while the production output does.  find_hdf5 picks whichever is
# actually there.
# The sample this section inspects.  Taken by role so it does not depend on a
# production naming its first signal set "nue"; section 3 reads it back to know
# which weight columns to expect.
H5_SAMPLE = next(n for n, c in SAMPLES.items() if c.get("kind") == "signal")
H5 = find_hdf5(H5_SAMPLE, SAMPLES)
TABLES = dump_tables(H5) if H5 else {}

In [ ]:
# To look closely at one table, e.g. the iLineFit column name (to confirm
# REGISTRY/ALTS picked the right one):
# dump_tables(H5, only=["iLineFit"])

## 3. Feature registry and consistency

Every variable is one row in `config/variables.json`, carrying **both** sides
of it: the `(HDF5 table, column)` training reads and the `(frame key, field)`
the tray reads.  `REGISTRY`, `ALTS`, `AUX` and `FEATURE_MAP` are views onto
that one table, so they cannot drift apart.

`check_registry()` asks whether those columns are really in *this* file —
column names differ between meta-project and pass versions, and whichever
variant is actually present is the one used.  `check_feature_map()` asks the
other question: do the two sides of a row still name the same quantity?  If
training reads one column and frame application another, the model produces
nonsense **without raising**.  Neither check needs icetray.

In [ ]:
from oscnext_l4 import data as l4data
from oscnext_l4.data import (REGISTRY, ALTS, AUX, NOISE_FEATURES, MUON_FEATURES,
                             WANTED, aux_for, scheme_of, check_registry,
                             check_feature_map)

print("--- noise BDT inputs (Table 11) ---")
check_registry(TABLES, NOISE_FEATURES)
print("\n--- muon BDT inputs (Table 12) ---")
check_registry(TABLES, MUON_FEATURES)

# Not every AUX column exists in every sample: noise_weight only in vuvuzela,
# OneWeight/PrimaryNeutrino* only in GENIE.  So ask only for the ones this
# sample's WEIGHT SCHEME needs.
#
# aux_for takes a SCHEME ("genie"), not a role ("signal").  Handing it a role
# returns an empty list, which check_registry used to report as a cheerful
# "found: 0/0" -- it now says outright that nothing was checked.  Passing the
# scheme lets it tell that mistake apart from detector data, which genuinely
# has no weight columns (its weight is 1/livetime, a property of the runs).
#
# TABLES is one sample's file, so this checks that one sample.
_scheme = scheme_of(SAMPLES[H5_SAMPLE])
print("\n--- weight columns (%s, scheme %s) ---" % (H5_SAMPLE, _scheme))
check_registry(TABLES, aux_for(_scheme), scheme=_scheme)

print()
check_feature_map()

## 4. HDF5 to numpy

Tables are matched on `Run/Event/SubEvent` — row order is not trusted.  If a
frame object is absent for some events, order-based reading **shifts** and
event A's `cog_z` ends up paired with event B's `NchCleaned`.

The weight denominator (`_n_files`) is the **number of L3 files**, read from
the `.meta.json` written next to each HDF5 — not the number of HDF5 files.


In [ ]:
from oscnext_l4.data import load_sample

# INCREMENTAL ON PURPOSE.  This cell used to start with `data = {}`, so
# re-running it -- to pick up a code change, or by reflex -- threw away
# everything already loaded and started over.  At pass2 that is hours: nue
# alone is 5.06M events across 68 parts.
#
# Set RELOAD = True to force a fresh read of everything, or delete one sample
# from `data` to reload just that one.
RELOAD = False

try:
    data
except NameError:
    data = {}
if RELOAD:
    data = {}

for name in SAMPLES:
    if name in data:
        print("%-8s already loaded: %d events (RELOAD=True to re-read)"
              % (name, len(data[name]["Run"])))
        continue
    d = load_sample(name, SAMPLES, WANTED)
    if d is not None:
        data[name] = d

print("\nLoaded:", {k: len(v["Run"]) for k, v in data.items()})
missing = [n for n in SAMPLES if n not in data]
if missing:
    print("[!] not loaded:", ", ".join(missing),
          "-- run section 1 for these, or check the HDF5 paths")

### Sanity check

If a variable is **entirely** NaN in a sample, that column was never booked —
feed it to training and the model silently ignores it.


In [ ]:
print("%-22s %s" % ("variable", "  ".join("%9s" % s for s in data)))
for f in NOISE_FEATURES + MUON_FEATURES:
    row, bad = [], False
    for d in data.values():
        v = d.get(f)
        frac = 100.0 * np.mean(~np.isfinite(v)) if v is not None else 100.0
        bad |= frac > 99.9
        row.append("%8.1f%%" % frac)
    print("%-22s %s%s" % (f, "  ".join("%9s" % x for x in row),
                          "  <-- ALWAYS MISSING" if bad else ""))
print("\n(Percentage of NaN.  A column at 100% was never booked.)")

## 5. Weights

Three separate notions:

1. **`w_phys` [Hz]** — the physical rate.  Used for distributions and cut
   performance.
2. **`weight`** — the training weight (section 6).  The note's preprocessing:
   equalise the class sums, then rescale into 0-1 (sec. 3.6.1).
3. **Unweighted counts** — statistical adequacy.

`add_weights` compares its result against Table 13 of the note (L3 rates) and
flags an order-of-magnitude discrepancy.

> `NORM`/`GAMMA` are **not** a real atmospheric flux, just a simple power law.
> Absolute rates will not match exactly; this is enough for shape comparison
> and for training.  A real flux needs `nuflux` (Honda) plus oscillations.


In [ ]:
from oscnext_l4.data import (add_weights, set_data_livetime,
                              livetime_from_headers, livetime_from_grl,
                              scheme_of)

# Detector data is weighted 1/livetime, and the livetime is not a column in
# anything we book -- it is a property of the runs.  So measure it from the
# event times, which ARE booked (I3EventHeader's MJD fields), before weighting.
#
# THIS IS AN ESTIMATE, NOT THE GOOD RUN LIST.  It is the span from the first to
# the last processed event of each run, summed over runs: it misses the head
# and tail (seconds, on runs of hours) and does not subtract dead time.  Quote
# a rate from it as such, and replace it with the GRL when that matters.
#
# Nothing downstream breaks if this is skipped -- build_dataset normalises each
# class to unit sum, so every constant gives identical TRAINING weights.  It
# changes whether w_phys is in Hz, which is what the rate cross-check and any
# comparison with Table 13 rest on.
_data_samples = [n for n in data
                 if n in SAMPLES and scheme_of(SAMPLES[n]) == "data"]
for _n in _data_samples:
    # BOTH are computed, and the Good Run List wins when it has every run.
    # Not because the estimate is wrong, but because only the GRL subtracts
    # dead time -- and running both prints the difference, which is the only
    # way to learn how good the estimate is on runs where the GRL is missing.
    print("=== livetime: from the event times (%s) ===" % _n)
    _t_hdr, _ = livetime_from_headers(data[_n]["_files"])

    print("\n=== livetime: from the Good Run List ===")
    _runs = np.unique(data[_n]["Run"])
    _t_grl, _found = livetime_from_grl(_runs)

    if len(_found) == len(_runs):
        print("\n  GRL / event-time span = %.4f   (the difference is dead "
              "time plus the head and tail the span misses)"
              % (_t_grl / _t_hdr if _t_hdr else float("nan")))
        set_data_livetime(_t_grl)
    else:
        print("\n  [!] the GRL does not cover every run -- falling back to "
              "the event-time span.")
        print("      A partial GRL total would divide these events by a "
              "livetime that does not cover them.")
        set_data_livetime(_t_hdr)

add_weights(data, SAMPLES)

In [ ]:
n = len(data)
fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 2.8))
axes = np.atleast_1d(axes)
for ax, (name, d) in zip(axes, data.items()):
    w = d["w_phys"]; w = w[np.isfinite(w) & (w > 0)]
    if w.size == 0:
        ax.set_title("%s: no weights" % name, fontsize=8); continue
    ax.hist(np.log10(w), bins=40, color="tab:blue")
    ax.set_title("%s\nmax/sum = %.1f%%" % (name, 100 * w.max() / w.sum()),
                 fontsize=8)
    ax.set_xlabel("log10(w_phys)", fontsize=7); ax.tick_params(labelsize=6)
plt.tight_layout(); plt.show()
print("max/sum above 5% means a single event dominates the rate.")

## 6. Training sets (`.npz`) and the train/test split

One `.npz` per classifier: the feature columns plus `label` (1 signal /
0 background), `weight` (training weight), `w_phys` (physical weight),
`istrain`, and `features` (the name order, so the model and the file cannot
drift apart).

**The training weight is computed here**: class sums are equalised, then all
weights are rescaled into `[0, 1]` (technical note sec. 3.6.1).

Two splitting traps are closed here:

1. **The signal split is drawn ONCE and shared by both classifiers.**  It used
   to be redrawn per classifier, so one event could be in the noise BDT's
   *training* set and the muon BDT's *test* set.  Since the L4 cut is the
   **conjunction** of the two, there was no common held-out set on which to
   evaluate the combined cut, and the final efficiency looked better than it
   was.
2. **CORSIKA is split by shower, not by event.**  The same air shower is
   reused `OverSampling` times; an event-level split spreads its copies across
   train and test and inflates the muon BDT's test efficiency.  The split is
   made on `Run`.  (Vuvuzela has no oversampling, so noise is split per event.)


In [ ]:
TRAIN_FRAC = 0.5


def stack(samples, features):
    """Concatenate several samples into one {name: array} dict."""
    out = {f: np.concatenate([data[s][f] for s in samples]) for f in features}
    for extra in ("w_phys", "Run"):
        out[extra] = np.concatenate([data[s][extra] for s in samples])
    return out


def report_missing_inputs(d, features, label):
    """
    Report events where at least one input is not finite.

    LightGBM routes missing values itself, so this is NOT a filter -- but we
    want to see how many events are affected and through which variable.  A
    variable that is 100% NaN stands out immediately here.
    """
    n = len(d["w_phys"])
    ok = np.ones(n, dtype=bool)
    for f in features:
        ok &= np.isfinite(np.asarray(d[f], dtype=np.float64))
    n_bad = int((~ok).sum())
    if n_bad:
        print("  [i] %-3s %d / %d events have a NaN input (%.2f%%)"
              % (label, n_bad, n, 100.0 * n_bad / n))
        for f in features:
            nf = int((~np.isfinite(np.asarray(d[f], dtype=np.float64))).sum())
            if nf:
                print("      %-22s %d" % (f, nf))
    return ok


def split_by_shower(runs, frac, rng_):
    """
    Train/test split at SHOWER level.

    In CORSIKA the same air shower is repeated OverSampling times; an
    event-level split puts copies of one shower on both sides and inflates the
    test efficiency.  Events sharing a `Run` always move together.
    """
    uniq = np.unique(runs)
    # A degenerate split is the danger here, and it is SILENT.  oscNext's
    # FixSimEventHeaders writes run_id = dataset_id, so a sample whose events
    # all come from one dataset has ONE distinct Run -- and this function would
    # then flip a single coin and send every event to train or every event to
    # test.  CORSIKA is safe (Run is the shower), MuonGun may not be.
    if len(uniq) < 20:
        print("  [!] split_by_shower: only %d distinct Run value(s) for %d "
              "events." % (len(uniq), len(runs)))
        print("      Too few to split on -- falling back to an EVENT-level "
              "split.  That is correct when the sample has no oversampling "
              "(MuonGun), and would leak copies if it does (CORSIKA).")
        return rng_.random(len(runs)) < frac
    keep = set(uniq[rng_.random(len(uniq)) < frac].tolist())
    return np.array([r in keep for r in runs], dtype=bool)


def build_dataset(tag, sig, bg, features, sig_istrain, bg_istrain):
    """Write one .npz for one classifier."""
    ws, wb = sig["w_phys"].copy(), bg["w_phys"].copy()
    for w in (ws, wb):
        w[~np.isfinite(w) | (w < 0)] = 0.0
    # equalise the class sums, then rescale into [0, 1]
    if ws.sum() > 0: ws /= ws.sum()
    if wb.sum() > 0: wb /= wb.sum()
    scale = max(ws.max(), wb.max())
    if scale > 0:
        ws /= scale; wb /= scale

    cols = {f: np.concatenate([np.asarray(sig[f], dtype=np.float64),
                               np.asarray(bg[f],  dtype=np.float64)])
            for f in features}
    cols["label"]   = np.r_[np.ones(len(ws)), np.zeros(len(wb))]
    cols["weight"]  = np.r_[ws, wb]
    cols["w_phys"]  = np.r_[np.asarray(sig["w_phys"], dtype=np.float64),
                            np.asarray(bg["w_phys"],  dtype=np.float64)]
    cols["istrain"] = np.r_[sig_istrain, bg_istrain]
    cols["features"] = np.array(features)

    path = os.path.join(DS_BASE, "L4_%s_dataset.npz" % tag)
    np.savez_compressed(path, **cols)
    print("  signal     train %7d / test %7d" % (sig_istrain.sum(), (~sig_istrain).sum()))
    print("  background train %7d / test %7d" % (bg_istrain.sum(), (~bg_istrain).sum()))
    print("  -> %s" % path)
    return path


# --- signal stacked ONCE, split drawn ONCE --------------------------------
ALL_FEATURES = sorted(set(NOISE_FEATURES) | set(MUON_FEATURES))

# Taken from the production table by ROLE, not by name -- the same rule the
# muon background uses below.  This used to be a hardcoded ["nue", "numu"],
# which silently dropped pass2's NuTau set even when it was loaded.  The
# production's own L4_model_data.py harvests GENIE 12xxxx, 14xxxx AND 16xxxx
# all as CLASSES["neutrino"], so including it is what the reference does.
SIGNAL = [n for n, c in SAMPLES.items()
          if c.get("kind") == "signal" and n in data]
print("signal sets: %s" % ", ".join(SIGNAL))

SIG = stack(SIGNAL, ALL_FEATURES)
SIG_ISTRAIN = rng.random(len(SIG["w_phys"])) < TRAIN_FRAC
print("signal: %d events, train %d / test %d  (BOTH classifiers use this split)"
      % (len(SIG_ISTRAIN), SIG_ISTRAIN.sum(), (~SIG_ISTRAIN).sum()))

In [ ]:
# ---------------------------------------------------------------------------
# THE NOISE CUT COMES BEFORE THE MUON BDT -- and that is the production's
# order, not a preference of ours.
# ---------------------------------------------------------------------------
# Technical note sec. 3.6.3: the muon classifier's background is detector data
# "following L4 noise cut", and that cut is WHY it is 99% muon.  The note's own
# Figures 19-24 carry `L4_NoiseClassifier_ProbNu > 0.7` in their cut box, and
# the fridge README says the same in its workflow: train noise, apply it, then
# train muon, "to avoid training using events that will be cut anyway".
#
# Training the muon BDT on the uncut population spends its capacity
# re-learning a rejection the noise BDT has already done, on events the L4 cut
# removes regardless.  This was CLAUDE.md open risk 5i.
NOISE_CUT = 0.70                     # the production's threshold

_noise_model = os.path.join(MODEL_DIR, "L4_noise_model.txt")
noise_prob = None

if os.path.exists(_noise_model):
    _nb_ = lgb.Booster(model_file=_noise_model)
    # The booster's own feature order, not NOISE_FEATURES: if the two ever
    # disagreed, indexing by our list would silently feed the columns in the
    # wrong order and the scores would be wrong without any error.
    _nfeat = list(_nb_.feature_name())

    def noise_prob(d):
        """P(neutrino) from the trained noise model, for a stacked dict."""
        X = np.column_stack([np.asarray(d[f], dtype=np.float64)
                             for f in _nfeat])
        return _nb_.predict(X)

    print("noise model : %s" % _noise_model)
    print("  features  : %s" % ", ".join(_nfeat))
    print("  cut       : P(nu) >= %.2f" % NOISE_CUT)
else:
    print("[!] no trained noise model at %s" % _noise_model)
    print("    Train it first (section 7 with STAGE='noise').  Without it the")
    print("    muon dataset below is built on the UNCUT population, which is")
    print("    NOT what the production trained on -- the cell says so again")
    print("    when it happens.")

In [ ]:
# Each classifier is skipped when its background is not loaded.  STAGE says
# which samples `select()` returned, so at STAGE="muon" there is no noise_bg
# and this half has nothing to build -- it used to run regardless and die in
# stack() with "need at least one array to concatenate".  The muon half below
# was already guarded; this is the same guard on the other side.
#
# DS_NOISE = None is not a loss: the resolver at the top of section 7 falls
# back to the .npz already on disk, so a muon-stage run still finds the noise
# dataset and model.
NOISE_BG = [n for n, c in SAMPLES.items()
            if c.get("kind") == "noise_bg" and n in data]

if not NOISE_BG:
    DS_NOISE = None
    print("=== noise BDT: SKIPPED -- no noise background loaded ===")
    print("    Expected at STAGE='muon'.  Use STAGE='noise' or 'all' to "
          "rebuild it.")
else:
    print("=== noise BDT  (signal = " + "+".join(SIGNAL)
          + ", background = " + "+".join(NOISE_BG) + ") ===")
    BG_NOISE = stack(NOISE_BG, ALL_FEATURES)
    report_missing_inputs(SIG, NOISE_FEATURES, "sig")
    report_missing_inputs(BG_NOISE, NOISE_FEATURES, "bg")
    # vuvuzela has no oversampling -> an event-level split is fine
    BG_NOISE_ISTRAIN = rng.random(len(BG_NOISE["w_phys"])) < TRAIN_FRAC
    DS_NOISE = build_dataset("noise", SIG, BG_NOISE, NOISE_FEATURES,
                             SIG_ISTRAIN, BG_NOISE_ISTRAIN)

# The muon BDT needs a muon background sample.  A pass2 noise-BDT run does not
# load one, so skip it instead of failing with a KeyError halfway through.
MUON_BG = [n for n, c in SAMPLES.items()
           if c.get("kind") == "muon_bg" and n in data]

# NEVER STACK SEVERAL MUON BACKGROUNDS.  pass2 has two -- MuonGun and real
# detector data -- and a model fitted on their union learns the difference
# between simulation and measurement as much as it learns the muon.  The
# production used data (note sec. 3.6.3: detector data "is considered more
# robust than using MuonGun MC at this processing level", and it is 99% muon
# after the noise cut), so data wins when both are loaded.
#
# This used to print MUON_BG[0] as "the background" while stack() took ALL of
# them -- the label said one sample and the data was another.  It could not
# bite while only one muon_bg sample existed; adding `data` made it live.
if len(MUON_BG) > 1:
    _pref = [n for n in MUON_BG if scheme_of(SAMPLES[n]) == "data"]
    print("  [!] %d muon backgrounds loaded (%s) -- they are NOT stacked."
          % (len(MUON_BG), ", ".join(MUON_BG)))
    MUON_BG = _pref[:1] or MUON_BG[:1]
    print("      using %r (the production's background is detector data)."
          % MUON_BG[0])

# NOTE, for a detector-data background: `Run` is the real run number, so there
# are only 18 distinct values and split_by_shower falls back to an event-level
# split.  That is defensible here -- separate muons in one run are independent,
# unlike the repeated copies of one CORSIKA shower the function exists for --
# but it is NOT what the production did: their test subsample selects runs by
# number.  With 18 runs a run-level split also risks losing the seasonal
# balance the run list was chosen for, so this is a real decision, not a
# detail.  Left as the event-level split, deliberately and visibly.
if not MUON_BG:
    DS_MUON = None
    print("\n=== muon BDT: SKIPPED -- no muon background sample loaded ===")
    print("    Load one: pass3 has 'corsika', pass2 has 'muongun'.")
else:
    print("\n=== muon BDT  (signal = " + "+".join(SIGNAL) + ", background = %s) ===" % MUON_BG[0])
    BG_MUON = stack(MUON_BG, ALL_FEATURES)

    # The noise cut, on BOTH classes -- see the cell above for why.
    #
    # The signal is masked together with its train/test flag, so a surviving
    # event keeps the side it was already on.  That is what preserves the
    # shared split (open risk 5c): the two classifiers are cut on jointly at
    # L4, so they need a COMMON held-out set, and re-drawing the split here
    # would destroy it.
    SIG_M, SIG_M_ISTRAIN = SIG, SIG_ISTRAIN
    if noise_prob is not None:
        _ks = noise_prob(SIG) >= NOISE_CUT
        _kb = noise_prob(BG_MUON) >= NOISE_CUT
        print("  noise cut P(nu) >= %.2f" % NOISE_CUT)
        print("    signal     %8d -> %8d  (%.1f%% kept)"
              % (len(_ks), int(_ks.sum()), 100.0 * _ks.mean()))
        print("    background %8d -> %8d  (%.1f%% kept)"
              % (len(_kb), int(_kb.sum()), 100.0 * _kb.mean()))
        SIG_M = {k: v[_ks] for k, v in SIG.items()}
        SIG_M_ISTRAIN = SIG_ISTRAIN[_ks]
        BG_MUON = {k: v[_kb] for k, v in BG_MUON.items()}
        if not _kb.sum():
            print("    [!] the cut removed EVERY background event.")
    else:
        print("  [!] NO NOISE CUT APPLIED -- the muon BDT below is trained on")
        print("      a population the production's never saw.  Train the noise")
        print("      model first; this is a real deviation, not a detail.")

    report_missing_inputs(SIG_M, MUON_FEATURES, "sig")
    report_missing_inputs(BG_MUON, MUON_FEATURES, "bg")
    # Split by Run: for CORSIKA that is the SHOWER, and one shower is repeated
    # OverSampling times, so an event-level split would spread its copies
    # across train and test.  For detector data Run is the real run and there
    # are only 18, below the function's threshold -- it falls back to an
    # event-level split and says so, which is defensible there because
    # separate muons in one run are independent.
    # Its OWN generator, not the shared `rng`.  The shared one is consumed in
    # order -- signal split, then noise background, then this -- so whether
    # the noise half above ran would change the numbers this draw receives,
    # and the same STAGE would not reproduce the same muon dataset.  The
    # signal split still comes from `rng` and is drawn first either way, which
    # is what keeps it common to both classifiers (open risk 5c).
    BG_MUON_ISTRAIN = split_by_shower(BG_MUON["Run"], TRAIN_FRAC,
                                      np.random.default_rng(RNG_SEED + 1))
    print("  %s: %d distinct Run values, %d events"
          % (MUON_BG[0], len(np.unique(BG_MUON["Run"])), len(BG_MUON["Run"])))
    DS_MUON = build_dataset("muon", SIG_M, BG_MUON, MUON_FEATURES,
                            SIG_M_ISTRAIN, BG_MUON_ISTRAIN)

## 7. Training

Calls `scripts/train_L4_classifier.py` — the training logic is not repeated
here so that there is only one implementation of it.

The hyperparameters are **Table 10 of the technical note**, fixed inside the
script.  Nothing needs to be passed by hand; only the occasional override such
as `--min-data-in-leaf` goes on the command line.

Watch for the `min_data_in_leaf` warning: Table 10 sets it to 500 against the
reference's much larger noise statistics.  With our sample that single setting
can decide the outcome, and the script says so up front.


In [ ]:
# KERNEL RESTARTED?  Then this cell is all that stands between section 0 and
# training.  The .npz files are already on disk, so training does not need
# sections 4-6 again -- and re-running them to recover a path string would
# reload millions of events and take far longer than the training itself.
#
# A dataset just built in section 6 always wins; this only fills in what the
# restart lost.  DS_BASE is suffixed per production, so a pass2 run can never
# pick up a pass3 .npz here.
for _tag in ("noise", "muon"):
    _var = "DS_" + _tag.upper()
    _path = os.path.join(DS_BASE, "L4_%s_dataset.npz" % _tag)
    if globals().get(_var):
        print("%-8s in memory : %s" % (_var, globals()[_var]))
    elif os.path.exists(_path):
        globals()[_var] = _path
        print("%-8s from disk : %s  (%.0f MB)"
              % (_var, _path, os.path.getsize(_path) / 1e6))
    else:
        globals()[_var] = None
        print("%-8s MISSING   : build it in section 6" % _var)

In [ ]:
def run_train(tag, dataset, extra=()):
    """
    Run the trainer and STREAM its output.

    This used to use subprocess.run(capture_output=True), which buffers
    everything and prints it only when the process exits.  That was invisible
    when training took seconds; on the pass2 sample it is 8.8M rows and many
    minutes, during which the cell showed nothing at all and a long run was
    indistinguishable from a hang.  `-u` on the child was already there; the
    parent was the one holding the output.
    """
    cmd = [sys.executable, "-u", TRAIN_PY, "--tag", tag,
           "--dataset", dataset, "--outdir", MODEL_DIR] + list(extra)
    print("$ " + " ".join(shlex.quote(c) for c in cmd) + "\n")
    t0 = time.time()
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    print("\n[%s] exit %d after %.1f s" % (tag, p.returncode, time.time() - t0))
    return p.returncode == 0


run_train("noise", DS_NOISE)

In [ ]:
if DS_MUON:
    run_train("muon", DS_MUON)
else:
    print("muon BDT skipped -- no muon background dataset (see section 6).")

## 8. Validation

`train_L4_classifier.py` wrote a `.json` metadata file and three plots per
model.

**Overtraining is judged by the train/test efficiency gap**, not by a KS test
on the score distributions.  Measured on this data, KS stamped the best model
"overtrained" and the worst one "clean" — with 165k signal events it catches
differences that are statistically significant and physically irrelevant.  The
gap compares what we actually care about: a model that memorised scores well
on train and badly on test.


In [ ]:
META = {}
for tag in ("noise", "muon"):
    p = os.path.join(MODEL_DIR, "L4_%s_model.json" % tag)
    if not os.path.exists(p):
        print("%-6s not trained yet" % tag)
        continue
    META[tag] = json.load(open(p))
    m = META[tag]["metrics"]
    gap = m.get("gap")
    print("%-6s %4d trees   eff %.1f%% at %.0f%% rejection  (%s bg events)   "
          "gap %s%s"
          % (tag, META[tag]["n_trees"],
             100 * (m["eff_at_target"] or float("nan")),
             100 * META[tag]["target_rejection"],
             m["bg_kept_at_target"],
             "%+.1f" % (100 * gap) if gap is not None else "?",
             "  <-- MEMORISING" if (gap or 0) > 0.05 else ""))
    if (m["bg_kept_at_target"] or 0) < 10:
        print("       [!] fewer than 10 background events define the target "
              "point -- not a measurement")

In [ ]:
from IPython.display import Image, display

# train_L4_classifier.py writes the plots as <tag>_<kind>.png
for tag in META:
    for kind in ("cuts", "overtrain", "dist"):
        p = os.path.join(MODEL_DIR, "%s_%s.png" % (tag, kind))
        if os.path.exists(p):
            print(tag, kind)
            display(Image(filename=p))
        else:
            print("%s %s: missing (%s)" % (tag, kind, p))

In [ ]:
# Feature importance -- which variable carries the model
for tag, meta in META.items():
    imp = meta["importance_gain"]
    tot = sum(imp.values()) or 1.0
    print("=== %s ===" % tag)
    for f, v in sorted(imp.items(), key=lambda x: -x[1]):
        print("  %-22s %8.1f  (%.1f%%)" % (f, v, 100 * v / tot))
    print()

## 9. Choosing the cut

The model output is LightGBM's `P(signal)` — the **same scale** the note uses,
so v00.07's `noise >= 0.70` / `muon >= 0.65` carry over directly.
`train_L4_classifier.py` already prints the efficiency and rejection at that
cut; the cell below is for picking from your own curve instead.

Reference targets (v00.07, pass2, Table 13):
- **noise**: 36.6 mHz down to <0.3 mHz, keeping ~96% of the neutrinos
- **muon**: 94% of the muons rejected, 87% of the neutrinos kept

The `table` entry in the `.json` holds the cut and the raw counts at each
rejection level — rows with fewer than 10 background events are not
measurements.


In [ ]:
CUTS = {}
for tag, meta in META.items():
    print("=== %s ===  (note cut: %s)" % (tag, meta["default_cut"]))
    print("%-10s %9s %10s %12s" % ("rejection", "eff", "cut", "background"))
    for row in meta["metrics"]["table"]:
        mark = "  <-- not a measurement" if row["bg_kept"] < 10 else ""
        print("%-9.1f%% %8.1f%% %10.4f %6d/%-6d%s"
              % (100 * row["rejection"], 100 * row["eff"], row["cut"],
                 row["bg_kept"], row["bg_total"], mark))
    CUTS[tag] = meta["default_cut"]
    print()

print("Cuts to use:", CUTS)

## 10. Applying the model to frames

To reprocess `.i3` files and write the classifier score into every frame, use
`oscnext_l4.classifier`:

```python
from oscnext_l4.classifier import add_L4_classifiers

add_L4_classifiers(tray, "L4_clf", model_dir=MODEL_DIR,
                   noise_cut=0.70, muon_cut=0.65)
```

That adds both models and writes the combined L4 cut
(`noise >= 0.70 AND muon >= 0.65`) into the frame as an `I3Bool`.

> The module reads its variables from the frame through `FEATURE_MAP`, while
> `REGISTRY` describes the HDF5 column training read.  Both are views onto the
> same row of `config/variables.json`, so they cannot drift apart — but the two
> sides of a row can still be made to name different quantities by hand, and
> then the model is wrong **without raising**.  The check below catches that
> (the same one as in section 3).


In [ ]:
check_feature_map()

## Checklist

**Processing**
- [ ] Smoke test clean, the stage breakdown is plausible (section 1)
- [ ] `check_registry` found every BDT input (section 3)
- [ ] `check_feature_map()` found no conflict (section 3)
- [ ] No variable is 100% NaN (section 4)

**Weights**
- [ ] No `.meta.json` warning (otherwise the divisor is wrong and the rates shift)
- [ ] `max/sum < 5%`
- [ ] Order of magnitude agrees with Table 13 (section 5)

**Training**
- [ ] `gap < 0.05` — no memorising (section 8)
- [ ] At least 10 background events define the target point (section 8)
- [ ] The `min_data_in_leaf` warning, if any, was taken into account (section 7)
- [ ] Cut chosen, efficiency/rejection in the right ballpark (section 9)

---

**Still unverified** — details in `CLAUDE.md` under *Open risks* and in
`docs/technical_note_comparison.md`:

- whether the VICH COG is charge weighted
- the reference time of `accumulated_time` (first pulse, or the trigger)
- `fill_ratio`'s `SphericalRadiusMean=1.6` was never re-tuned for oscNext —
  and that variable carries 61% of the noise model's gain, making it the
  single highest-value knob
- the muon BDT background is CORSIKA rather than real data, so there is no
  data/MC check
- no nutau set, so ~3% of the signal is missing
- the noise MC statistics are thin (~2000 events): nothing right of 99%
  rejection is measurable
